## Importation des Bibliothèques

In [ ]:
import kagglehub
import os
import yaml
import shutil
import torch
from sklearn.model_selection import train_test_split
from collections import Counter

## Téléchargement du dataset

In [ ]:
PATH = kagglehub.dataset_download(
    "ashfakyeafi/road-vehicle-images-dataset"
)

Using Colab cache for faster access to the 'road-vehicle-images-dataset' dataset.


## Définir les chemins du dataset

In [ ]:
DATASET_DIR = os.path.join(PATH, "trafic_data")

TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR = os.path.join(DATASET_DIR, "valid")

TRAIN_IMAGES_DIR = os.path.join(TRAIN_DIR, "images")
TRAIN_LABELS_DIR = os.path.join(TRAIN_DIR, "labels")

VAL_IMAGES_DIR = os.path.join(VAL_DIR, "images")
VAL_LABELS_DIR = os.path.join(VAL_DIR, "labels")

DATA_YAML_PATH = os.path.join(DATASET_DIR, "data_1.yaml")

## Affichage des chemins et vérification du dataset

In [ ]:
paths = {
    "Train images": TRAIN_IMAGES_DIR,
    "Train labels": TRAIN_LABELS_DIR,
    "Validation images": VAL_IMAGES_DIR,
    "Validation labels": VAL_LABELS_DIR,
    "Data YAML": DATA_YAML_PATH
}

for name, path in paths.items():
    status = "OK" if os.path.exists(path) else "INTROUVABLE"
    print(f"{name} : {path} [{status}]")

Train images : /kaggle/input/road-vehicle-images-dataset/trafic_data/train/images [OK]
Train labels : /kaggle/input/road-vehicle-images-dataset/trafic_data/train/labels [OK]
Validation images : /kaggle/input/road-vehicle-images-dataset/trafic_data/valid/images [OK]
Validation labels : /kaggle/input/road-vehicle-images-dataset/trafic_data/valid/labels [OK]
Data YAML : /kaggle/input/road-vehicle-images-dataset/trafic_data/data_1.yaml [OK]


## Exploration de la structure du dataset

In [ ]:
for root, dirs, files in os.walk(PATH):
    level = root.replace(PATH, "").count(os.sep)

    if level <= 3:
        print("    " * level + os.path.basename(root) + "/")

        for file in files[:5]:
            print("    " * (level + 1) + file)

road-vehicle-images-dataset/
    trafic_data/
        data_1.yaml
        valid/
            labels.cache
            labels/
                Pias--299-_jpg.rf.d99789045f9e78094450214f892c5224.txt
                Pias--350-_jpg.rf.66e704ee7974871a7e7a9b9767b648cc.txt
                Pias--366-_jpg.rf.67e15b0ae10be95e51aba283b94d074a.txt
                Pias--253-_jpg.rf.b175d2bc3e0734e8ccf295012ae05e11.txt
                Pias--228-_jpg.rf.66a1a43811de3d097d2c4c0aaf6164cf.txt
            images/
                Pias--38-_jpg.rf.22e718dc7c4e90bfd00a4a5985e1e549.jpg
                Pias--34-_jpg.rf.1d1c2750781e3c078ec8d754b9a88ddf.jpg
                Pias--254-_jpg.rf.730db135b09671b117629123a18a50dc.jpg
                Pias--245-_jpg.rf.da3fd014f9fa2ccd9b9edbe8d4d85a4b.jpg
                Pias--31-_jpg.rf.ba2097448252f42d63ec8b2073ee4ee6.jpg
        train/
            labels.cache
            labels/
                Dipto_429_jpg.rf.41052bcdafb61f4810afcbcbb807ab4b.txt
                D

## Comptage des images et des annotations

In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def count_images(directory):
    return sum(
        1 for file in os.listdir(directory)
        if file.lower().endswith(IMAGE_EXTENSIONS)
    )

def count_labels(directory):
    return sum(
        1 for file in os.listdir(directory)
        if file.lower().endswith(".txt")
    )

print("Nombre d'images Train :", count_images(TRAIN_IMAGES_DIR))
print("Nombre de labels Train :", count_labels(TRAIN_LABELS_DIR))

print("Nombre d'images Validation :", count_images(VAL_IMAGES_DIR))
print("Nombre de labels Validation :", count_labels(VAL_LABELS_DIR))

Nombre d'images Train : 2704
Nombre de labels Train : 2704
Nombre d'images Validation : 300
Nombre de labels Validation : 300


## Lecture du fichier de configuration YAML

In [ ]:
with open(DATA_YAML_PATH, "r") as file:
    print(file.read())

train: ../train/images
val: ../valid/images

nc: 21
names: ['ambulance', 'army vehicle', 'auto rickshaw', 'bicycle', 'bus', 'car', 'garbagevan', 'human hauler', 'minibus', 'minivan', 'motorbike', 'pickup', 'policecar', 'rickshaw', 'scooter', 'suv', 'taxi', 'three wheelers -CNG-', 'truck', 'van', 'wheelbarrow']


## Validation des annotations

In [ ]:
def validate_labels(labels_dir, nc):
    invalid_files = []
    invalid_lines = []
    empty_files = 0
    total_objects = 0

    for filename in os.listdir(labels_dir):
        if not filename.endswith(".txt"):
            continue

        file_path = os.path.join(labels_dir, filename)

        with open(file_path, "r") as file:
            lines = file.readlines()

        if not lines:
            empty_files += 1
            continue

        for line_number, line in enumerate(lines, start=1):
            values = line.strip().split()

            if len(values) != 5:
                invalid_lines.append((filename, line_number, "Nombre de valeurs incorrect"))
                continue

            try:
                class_id = int(values[0])
                x_center, y_center, width, height = map(float, values[1:])
            except ValueError:
                invalid_lines.append((filename, line_number, "Valeurs non numériques"))
                continue

            if not 0 <= class_id < nc:
                invalid_lines.append((filename, line_number, "Class ID invalide"))
                continue

            if not all(0 <= value <= 1 for value in [x_center, y_center, width, height]):
                invalid_lines.append((filename, line_number, "Coordonnées hors limites"))
                continue

            if width == 0 or height == 0:
                invalid_lines.append((filename, line_number, "Bounding box de taille nulle"))
                continue

            total_objects += 1

    print("Dossier :", labels_dir)
    print("Objets valides :", total_objects)
    print("Fichiers d'annotations vides :", empty_files)
    print("Lignes invalides :", len(invalid_lines))

    for error in invalid_lines[:10]:
        print(error)

    return invalid_files, invalid_lines


train_errors = validate_labels(TRAIN_LABELS_DIR, 21)
val_errors = validate_labels(VAL_LABELS_DIR, 21)

Dossier : /kaggle/input/road-vehicle-images-dataset/trafic_data/train/labels
Objets valides : 21764
Fichiers d'annotations vides : 2
Lignes invalides : 0
Dossier : /kaggle/input/road-vehicle-images-dataset/trafic_data/valid/labels
Objets valides : 2560
Fichiers d'annotations vides : 0
Lignes invalides : 24
('Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt', 1, 'Bounding box de taille nulle')
('Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt', 2, 'Bounding box de taille nulle')
('Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt', 3, 'Bounding box de taille nulle')
('Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt', 4, 'Bounding box de taille nulle')
('Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt', 5, 'Bounding box de taille nulle')
('Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt', 6, 'Bounding box de taille nulle')
('Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt', 7, 'Bounding box de taille nulle')
('Pias--359-_PNG.rf.43bcf36efe5cf8

## Analyse détaillée des erreurs

In [ ]:
def inspect_invalid_labels(labels_dir, nc):
    error_files = Counter()
    error_types = Counter()

    for filename in os.listdir(labels_dir):
        if not filename.endswith(".txt"):
            continue

        file_path = os.path.join(labels_dir, filename)

        with open(file_path, "r") as file:
            lines = file.readlines()

        for line_number, line in enumerate(lines, start=1):
            values = line.strip().split()

            if len(values) != 5:
                error_files[filename] += 1
                error_types["Nombre de valeurs incorrect"] += 1
                continue

            try:
                class_id = int(values[0])
                x, y, width, height = map(float, values[1:])
            except ValueError:
                error_files[filename] += 1
                error_types["Valeurs non numériques"] += 1
                continue

            if not 0 <= class_id < nc:
                error_files[filename] += 1
                error_types["Class ID invalide"] += 1
            elif not all(0 <= value <= 1 for value in [x, y, width, height]):
                error_files[filename] += 1
                error_types["Coordonnées hors limites"] += 1
            elif width <= 0 or height <= 0:
                error_files[filename] += 1
                error_types["Bounding box de taille nulle"] += 1

    print("Types d'erreurs :")
    for error_type, count in error_types.items():
        print(f"{error_type} : {count}")

    print("\nFichiers concernés :")
    for filename, count in error_files.items():
        print(f"{filename} : {count} ligne(s) invalide(s)")

inspect_invalid_labels(VAL_LABELS_DIR, 21)

Types d'erreurs :
Bounding box de taille nulle : 24

Fichiers concernés :
Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt : 14 ligne(s) invalide(s)
Pias--360-_PNG.rf.8405b0e44009a9300e0a1100ccf7d5b3.txt : 10 ligne(s) invalide(s)


## Nettoyage et séparation Train/Test

In [ ]:
OUTPUT_DIR = "/kaggle/working/roadvision"

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

# Préparer les dossiers
for split in ["train", "test", "valid"]:
    for folder in ["images", "labels"]:
        os.makedirs(os.path.join(OUTPUT_DIR, split, folder))

# Exclure les deux annotations incorrectes de validation
invalid_stems = {
    "Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60",
    "Pias--360-_PNG.rf.8405b0e44009a9300e0a1100ccf7d5b3"
}

# Séparer le train en train et test
images = [
    f for f in os.listdir(TRAIN_IMAGES_DIR)
    if os.path.isfile(os.path.join(TRAIN_IMAGES_DIR, f))
    and os.path.splitext(f)[0] + ".txt" in os.listdir(TRAIN_LABELS_DIR)
]

train_images, test_images = train_test_split(
    images, test_size=0.15, random_state=42
)

# Copier les fichiers train et test
for split, files in [("train", train_images), ("test", test_images)]:
    for image_name in files:
        stem = os.path.splitext(image_name)[0]
        shutil.copy2(
            os.path.join(TRAIN_IMAGES_DIR, image_name),
            os.path.join(OUTPUT_DIR, split, "images", image_name)
        )
        shutil.copy2(
            os.path.join(TRAIN_LABELS_DIR, stem + ".txt"),
            os.path.join(OUTPUT_DIR, split, "labels", stem + ".txt")
        )

# Copier validation en excluant les deux images
for image_name in os.listdir(VAL_IMAGES_DIR):
    stem = os.path.splitext(image_name)[0]

    if stem in invalid_stems:
        continue

    label_name = stem + ".txt"
    image_path = os.path.join(VAL_IMAGES_DIR, image_name)
    label_path = os.path.join(VAL_LABELS_DIR, label_name)

    if os.path.exists(label_path):
        shutil.copy2(
            image_path, os.path.join(OUTPUT_DIR, "valid", "images", image_name)
        )
        shutil.copy2(
            label_path, os.path.join(OUTPUT_DIR, "valid", "labels", label_name)
        )

print("Train :", len(train_images))
print("Test :", len(test_images))
print("Validation :", len(os.listdir(os.path.join(OUTPUT_DIR, "valid", "images"))))

Train : 2298
Test : 406
Validation : 298


## Configuration du dataset

In [ ]:
DATASET_DIR = "/kaggle/working/roadvision"

data = {
    "train": os.path.join(DATASET_DIR, "train/images"),
    "val": os.path.join(DATASET_DIR, "valid/images"),
    "test": os.path.join(DATASET_DIR, "test/images"),
    "nc": 21,
    "names": [
        "ambulance", "army vehicle", "auto rickshaw", "bicycle",
        "bus", "car", "garbagevan", "human hauler", "minibus",
        "minivan", "motorbike", "pickup", "policecar", "rickshaw",
        "scooter", "suv", "taxi", "three wheelers -CNG-",
        "truck", "van", "wheelbarrow"
    ]
}

DATA_YAML_PATH = os.path.join(DATASET_DIR, "data.yaml")

with open(DATA_YAML_PATH, "w") as file:
    yaml.dump(data, file, sort_keys=False)

print("Fichier créé :", DATA_YAML_PATH)
print(open(DATA_YAML_PATH).read())

Fichier créé : /kaggle/working/roadvision/data.yaml
train: /kaggle/working/roadvision/train/images
val: /kaggle/working/roadvision/valid/images
test: /kaggle/working/roadvision/test/images
nc: 21
names:
- ambulance
- army vehicle
- auto rickshaw
- bicycle
- bus
- car
- garbagevan
- human hauler
- minibus
- minivan
- motorbike
- pickup
- policecar
- rickshaw
- scooter
- suv
- taxi
- three wheelers -CNG-
- truck
- van
- wheelbarrow



## Télécharger le dépôt officiel YOLOv9

In [ ]:
REPO_DIR = "/kaggle/working/yolov9"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/WongKinYiu/yolov9.git {REPO_DIR}

%cd /kaggle/working/yolov9

!pip install -q -r requirements.txt

Cloning into '/kaggle/working/yolov9'...
remote: Enumerating objects: 781, done.
remote: Total 781 (delta 0), reused 0 (delta 0), pack-reused 781 (from 1)
Receiving objects: 100% (781/781), 3.27 MiB | 23.92 MiB/s, done.
Resolving deltas: 100% (330/330), done.
/kaggle/working/yolov9


## Télécharger les poids pré-entraînés

In [ ]:
FILE_PATH = "/kaggle/working/yolov9/train_dual.py"

with open(FILE_PATH, "r") as file:
    code = file.read()

code = code.replace(
    "torch.load(weights, map_location='cpu')",
    "torch.load(weights, map_location='cpu', weights_only=False)"
)

with open(FILE_PATH, "w") as file:
    file.write(code)

# Vérification
with open(FILE_PATH, "r") as file:
    for i, line in enumerate(file, 1):
        if "torch.load(weights" in line:
            print(f"Ligne {i} :", line.strip())

Ligne 110 : ckpt = torch.load(weights, map_location='cpu', weights_only=False)  # load checkpoint to CPU to avoid CUDA memory leak


## Vérification de l’environnement

In [ ]:
REPO_DIR = "/kaggle/working/yolov9"
DATASET_DIR = "/kaggle/working/roadvision"

print("GPU disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

files = [
    os.path.join(REPO_DIR, "train_dual.py"),
    os.path.join(REPO_DIR, "models/detect/yolov9-c.yaml"),
    os.path.join(REPO_DIR, "data/hyps/hyp.scratch-high.yaml"),
    os.path.join(REPO_DIR, "yolov9-c.pt"),
    os.path.join(DATASET_DIR, "data.yaml")
]

for file in files:
    print("OK" if os.path.exists(file) else "MANQUANT", ":", file)

for split in ["train", "valid", "test"]:
    path = os.path.join(DATASET_DIR, split, "images")
    print(f"{split} :", len(os.listdir(path)) if os.path.exists(path) else "DOSSIER MANQUANT")

GPU disponible : True
GPU : Tesla T4
OK : /kaggle/working/yolov9/train_dual.py
OK : /kaggle/working/yolov9/models/detect/yolov9-c.yaml
OK : /kaggle/working/yolov9/data/hyps/hyp.scratch-high.yaml
OK : /kaggle/working/yolov9/yolov9-c.pt
OK : /kaggle/working/roadvision/data.yaml
train : 2298
valid : 298
test : 406


## Entraînement de YOLOv9-C

In [ ]:
%cd /kaggle/working/yolov9

!python train_dual.py \
    --workers 2 \
    --device 0 \
    --batch 4 \
    --data /kaggle/working/roadvision/data.yaml \
    --img 640 \
    --cfg models/detect/yolov9-c.yaml \
    --weights yolov9-c.pt \
    --name roadvision-yolov9c \
    --hyp data/hyps/hyp.scratch-high.yaml \
    --epochs 10

/kaggle/working/yolov9
/kaggle/working/yolov9/utils/general.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg
I0000 00:00:1793314410.134573    5805 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793314412.583119    5805 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: WARNING W&B disabled due to login timeout.
wandb: W&B API key is config

In [ ]:
FILE_PATH = "/kaggle/working/yolov9/utils/general.py"

with open(FILE_PATH, "r") as file:
    code = file.read()

code = code.replace(
    "torch.load(f, map_location=torch.device('cpu'))",
    "torch.load(f, map_location=torch.device('cpu'), weights_only=False)"
)

with open(FILE_PATH, "w") as file:
    file.write(code)

print("Correction appliquée.")

Correction appliquée.


In [ ]:
%cd /kaggle/working/yolov9

from utils.general import strip_optimizer

WEIGHTS_DIR = "runs/train/roadvision-yolov9c3/weights"

for name in ["best.pt", "last.pt"]:
    path = os.path.join(WEIGHTS_DIR, name)

    if os.path.exists(path):
        strip_optimizer(path)
        print("Checkpoint sauvegardé :", path)
    else:
        print("Fichier introuvable :", path)

/kaggle/working/yolov9


/kaggle/working/yolov9/utils/general.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg
Optimizer stripped from runs/train/roadvision-yolov9c3/weights/best.pt, 102.8MB


Checkpoint sauvegardé : runs/train/roadvision-yolov9c3/weights/best.pt


Optimizer stripped from runs/train/roadvision-yolov9c3/weights/last.pt, 102.8MB


Checkpoint sauvegardé : runs/train/roadvision-yolov9c3/weights/last.pt


## Évaluation sur le Test Set

In [ ]:
%cd /kaggle/working/yolov9

!python val_dual.py \
    --data /kaggle/working/roadvision/data.yaml \
    --weights /kaggle/working/yolov9/runs/train/roadvision-yolov9c3/weights/best.pt \
    --img 640 \
    --batch-size 4 \
    --conf-thres 0.001 \
    --device 0 \
    --task test \
    --verbose \
    --name roadvision-test

/kaggle/working/yolov9
/kaggle/working/yolov9/utils/general.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg
val_dual: data=/kaggle/working/roadvision/data.yaml, weights=['/kaggle/working/yolov9/runs/train/roadvision-yolov9c3/weights/best.pt'], batch_size=4, imgsz=640, conf_thres=0.001, iou_thres=0.7, max_det=300, task=test, device=0, workers=8, single_cls=False, augment=False, verbose=True, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=runs/val, name=roadvision-test, exist_ok=False, half=False, dnn=False, min_items=0
YOLO 🚀 v0.1-104-g5b1ea9a Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)

Traceback (most recent call last):
  File "/kaggle/working/yolov9/val_dual.py", line 393, in <module>
    main(opt)
    ~~~~^^^